# RAG Minecraft

## Configuration Gemini API

In [1]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import format_document
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import Chroma
import shutil
import os
import csv
import requests
from urllib.parse import unquote
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter
from IPython.display import Markdown
import cloudscraper
import time


C:\Users\26652\AppData\Local\Temp\ipykernel_29872\2324973339.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
from api_config import configure_google_api_key

GOOGLE_API_KEY = configure_google_api_key()

## Scrapping

Congig sources

In [3]:
WIKI_PAGES = [
    "Minecraft"
]

FANDOM_URLS = [
    "https://minecraft.fandom.com/fr/wiki/Survie",
    "https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atif",
    "https://minecraft.fandom.com/fr/wiki/Hardcore",
    "https://minecraft.fandom.com/fr/wiki/Fabrication",
    "https://minecraft.fandom.com/fr/wiki/Cuisson",
    "https://minecraft.fandom.com/fr/wiki/Alchimie",
    "https://minecraft.fandom.com/fr/wiki/La_Foire_aux_Questions",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/Choses_%C3%A0_ne_PAS_faire",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/Survie_dans_le_Nether",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/L%27End_et_l%27Ender_Dragon",
    "https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atures"
]

In [4]:
scraper = cloudscraper.create_scraper()

In [5]:
def write_csv(file_name, paragraphs):
    with open(file_name, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["text"])
        for p in paragraphs:
            writer.writerow([p])

In [6]:
def write_csv(file_name, paragraphs):
    # Create parent folder automatically if it does not exist
    folder = os.path.dirname(file_name)
    if folder:
        os.makedirs(folder, exist_ok=True)

    with open(file_name, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["text"])
        for p in paragraphs:
            writer.writerow([p])

Wikipedia

In [7]:
def scrape_wikipedia(title: str):

    url = "https://fr.wikipedia.org/w/api.php"

    params = {
        "action": "parse",
        "page": title,
        "format": "json",
        "prop": "text",
        "redirects": True
    }

    r = requests.get(url, params=params, headers={"User-Agent": "Mozilla/5.0"})
    data = r.json()

    html = data["parse"]["text"]["*"]
    soup = BeautifulSoup(html, "html.parser")

    texts = []
    capture = False

    for tag in soup.find_all(["h2", "p"]):

        t = tag.get_text(" ", strip=True)

        if tag.name == "h2" and "Références" in t:
            break

        if tag.name == "h2":
            capture = True
            continue

        if capture and t:
            texts.append(t)

    write_csv("files/wikipedia.csv", texts)

    return Document(
        page_content="\n\n".join(texts),
        metadata={"source": f"wikipedia:{title}"}
    )

Fandom

In [8]:
def scrape_fandom(url: str):

    headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    ),
    "Accept-Language": "fr-FR,fr;q=0.9",
    "Referer": "https://www.google.com/"
    }

    r = scraper.get(url, headers=headers)

    soup = BeautifulSoup(r.text, "html.parser")

    content = soup.find("div", {"class": "mw-parser-output"})
    if not content:
        return None

    texts = []

    for tag in content.find_all(["p", "h2", "h3", "li"]):

        t = tag.get_text(" ", strip=True)

        if not t:
            continue

        if len(t) < 20:
            continue

        if "Voir aussi" in t or "Notes et références" in t:
            break

        texts.append(t)

    paragraphs = [p.strip() for p in texts if p.strip()]
    page_name = url.split("/wiki/")[-1]
    page_name = unquote(page_name)
    file_name = f"files/{page_name}.csv"

    write_csv(file_name, paragraphs)


    return Document(
        page_content="\n\n".join(texts),
        metadata={"source": url}
    )

Build dataset

In [9]:
docs = []

# Wikipedia
for page in WIKI_PAGES:
    try:
        docs.append(scrape_wikipedia(page))
        print("WIKI OK:", page)
    except Exception as e:
        print("WIKI ERROR:", page, e)

# Fandom
for url in FANDOM_URLS:
    try:
        doc = scrape_fandom(url)
        if doc:
            docs.append(doc)
            print("FANDOM OK:", url)
    except Exception as e:
        print("FANDOM ERROR:", url, e)

print("\nTOTAL DOCS:", len(docs))

WIKI OK: Minecraft
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Survie
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atif
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Hardcore
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Fabrication
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Cuisson
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Alchimie
FANDOM OK: https://minecraft.fandom.com/fr/wiki/La_Foire_aux_Questions
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Tutoriels/Choses_%C3%A0_ne_PAS_faire
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Tutoriels/Survie_dans_le_Nether
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Tutoriels/L%27End_et_l%27Ender_Dragon
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atures

TOTAL DOCS: 12


Chunking

In [10]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=150
)

split_docs = splitter.split_documents(docs)

print("CHUNKS:", len(split_docs))

CHUNKS: 105


Enrchich

In [11]:
def enrich_with_source(docs):
    enriched = []

    for d in docs:
        src = d.metadata.get("source", "unknown")

        source_type = "WIKIPEDIA" if "wikipedia" in src else "FANDOM"

        d.page_content = (
            f"SOURCE: {src}\n"
            f"SOURCE_TYPE: {source_type}\n\n"
            f"{d.page_content}"
        )

        enriched.append(d)

    return enriched

split_docs = enrich_with_source(split_docs)

## Initialize embedding model + Retriever

In [12]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

gemini_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="retrieval_document"
)

#### Store data using chroma

In [13]:
import shutil
import os
path = "./chroma_minecraft_db"

if os.path.exists(path):
    try:
        shutil.rmtree(path)
    except PermissionError:
        print("dossier encore utilisé → restart kernel puis relance")

In [ ]:
persist_dir = "./chroma_minecraft_db"

# 1. Création + stockage (UNE SEULE FOIS)
vectorstore = Chroma(
    persist_directory=persist_dir,
    embedding_function=gemini_embeddings
)
batch_size = 10

for i in range(0, len(split_docs), batch_size):
    batch = split_docs[i:i+batch_size]
    vectorstore.add_documents(batch)
    time.sleep(10)

# 2. Reload propre (pour usage RAG)
vectorstore_disk = Chroma(
    persist_directory=persist_dir,
    embedding_function=gemini_embeddings
)

# 3. Retriever FINAL utilisé par le RAG
retriever = vectorstore_disk.as_retriever(
    search_kwargs={"k": 6}
)

C:\Users\26652\AppData\Local\Temp\ipykernel_29872\2364975380.py:4: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [15]:
# Check the number of chunks generated after text splitting.
print("split_docs:", len(split_docs))

# Check the number of documents actually stored in the Chroma vector database.
# If this number is equal to len(split_docs), the database was built without missing or duplicated chunks.
print("chroma:", vectorstore._collection.count())

split_docs: 105
chroma: 105


#### Create a retriever using Chroma

In [16]:
print(retriever.invoke("Minecraft")[0].page_content[:500])

def retrieve_documents(retriever, query):
    if hasattr(retriever, "invoke"):
        return retriever.invoke(query)
    if hasattr(retriever, "get_relevant_documents"):
        return retriever.get_relevant_documents(query)
    raise AttributeError("Le retriever ne supporte ni invoke ni get_relevant_documents.")

SOURCE: https://minecraft.fandom.com/fr/wiki/La_Foire_aux_Questions
SOURCE_TYPE: FANDOM

Q: Je meurs tout le temps dès la première nuit ! Aidez-moi ! [ ]

Q: Comment changer l'apparence de mon personnage ? [ ]

Q: Le niveau de l'eau peut-il monter ? [ ]

Q: Quelles sont les touches ? [ ]

Q: Comment jeter d'un coup une pile d'objets ? [ ]

Q: Comment sauvegarder/charger ma position en multijoueur ? [ ]

Q: Comment faire pousser des plantes ? Elles ne font que disparaître… [ ]

Q: Pourquoi est-ce


## Generator

#### Initialize Generator

In [17]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

#### Create prompt templates

In [18]:
# Prompt template to query Gemini
llm_prompt_template = """You are an assistant for question-answering tasks.
Use the following context to answer the question.
If you don't know the answer, just say that you don't know.
Use five sentences maximum and keep the answer concise.\n
Question: {question} \nContext: {context} \nAnswer:"""

llm_prompt = PromptTemplate.from_template(llm_prompt_template)

print(llm_prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} template="You are an assistant for question-answering tasks.\nUse the following context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse five sentences maximum and keep the answer concise.\n\nQuestion: {question} \nContext: {context} \nAnswer:"


#### Create a stuff documents chain

In [20]:
# Combine data from documents to readable string format.
def format_docs(docs):
    formatted_docs = []

    for doc in docs:
        source = doc.metadata.get("source", "unknown")
        content = doc.page_content

        formatted_text = f"[{source}]\n{content}"

        formatted_docs.append(formatted_text)

    return "\n\n".join(formatted_docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | llm_prompt
    | llm
    | StrOutputParser()
)

### Prompt the model

In [21]:
Markdown(rag_chain.invoke("Quels sont les principaux modes de jeu dans Minecraft ?"))

D'après les documents fournis, les principaux modes de jeu dans Minecraft sont le **mode Créatif** et le **mode Survie**. 

Le mode Créatif est axé sur la construction libre, offrant des ressources illimitées et la possibilité de voler sans craindre les dégâts ou la mort. Le mode Survie, quant à lui, oblige les joueurs à récolter des ressources, fabriquer des outils, gérer leur faim et combattre des monstres pour prospérer. Les textes mentionnent également le **mode Hardcore**, une variante du mode Survie où la difficulté est bloquée sur "Difficile" et où le joueur ne dispose d'une unique vie.

In [22]:
Markdown(rag_chain.invoke("Comment survivre dans le Nether dans Minecraft ?"))

Pour survivre dans le Nether, construisez d'abord un abri solide en pierre taillée autour de votre portail pour le protéger des tirs de Ghasts. Pensez à emporter un briquet pour pouvoir rallumer votre portail en cas de besoin, ainsi qu'un établi, des outils et des provisions. Ne placez surtout pas de lit, car celui-ci explosera si vous essayez d'y dormir. Pour ne pas vous perdre, marquez toujours votre chemin à l'aide de torches ou de blocs de pierre bien visibles. Enfin, lors de l'exploration des forteresses, placez de nombreuses torches autour des générateurs de Blazes pour limiter leur apparition.

In [23]:
Markdown(rag_chain.invoke("C'est quoi l'alchimie dans Minecraft?"))

L'alchimie (ou distillation) dans Minecraft est la science qui permet de créer des potions, des potions jetables et des potions persistantes. Pour la pratiquer, le joueur doit utiliser un alambic alimenté par de la poudre de Blaze. Le processus consiste à distiller un ingrédient placé dans l'emplacement supérieur dans des fioles situées en bas. La verrue du Nether est le principal ingrédient de base requis pour fabriquer la majeure partie de ces potions.

In [24]:
print(format_docs(retriever.invoke("Alchimie")))

[https://minecraft.fandom.com/fr/wiki/Alchimie]
SOURCE: https://minecraft.fandom.com/fr/wiki/Alchimie
SOURCE_TYPE: FANDOM

Diagramme d'alchimie des potions (recettes les plus efficaces, hors potions jetables et persistantes)

L' alchimie (ou distillation) (nom anglais : brewing ) est la science permettant de créer des potions , des potions jetables des potions persistantes dans Minecraft.

1 Guide général de l'alchimiste

2 Outils d'alchimiste

3 Ingrédients 3.1 Ingrédients de base et modificateurs 3.2 Ingrédients à effet

3.1 Ingrédients de base et modificateurs

3.2 Ingrédients à effet

4 Potions 4.1 Potions de base 4.2 Potions avec effets 4.2.1 Positives 4.2.2 Négatives 4.2.3 Mixtes

4.2 Potions avec effets 4.2.1 Positives 4.2.2 Négatives 4.2.3 Mixtes

Guide général de l'alchimiste [ ]

Une grille d'alchimie.

Le joueur doit tout d'abord fabriquer un alambic , le poser sur le sol, puis faire un clic droit dessus pour afficher la grille d'alchimie. L'emplacement du haut est prévu pou